This Notebook demonstrates how to build a **Retrieval-Augmented Generation (RAG)** pipeline to perform Information Retrieval (IR) over custom data like CSV 

# Document Loading
**What happens:** The CSVLoader reads the raw file (OutdoorClothingCatalog_1000.csv) and splits each row into a distinct Document object containing its text (page_content) and metadata.

**Why it matters:** Standard LLMs only know their training data. Loading documents converts external data into a format usable by a retrieval system.

# Generating Text Embeddings
**What happens:** Using OpenAIEmbeddings, textual data (e.g., product descriptions) is converted into high-dimensional numerical vectors (lists of 1,536 numbers).

**Why it matters:** Embeddings capture semantic meaning. Phrases with similar meanings end up close together in vector space, allowing searches based on concepts rather than exact keyword matches.

# Storing in a Vector Database
**What happens:** The embeddings and their corresponding document texts are indexed using an in-memory vector store (DocArrayInMemorySearch).

**Why it matters:** The vector store acts as the retrieval database, allowing fast similarity searches over thousands of document vectors.

# Vector Similarity Search (Retrieval Step)
**What happens**: When given a prompt like "Please suggest a shirt with sunblocking", the pipeline:

* 1.Converts the user's query into a vector embedding.
* 2.Measures vector similarity between the query vector and the stored document vectors.
* 3.Returns the top $k$ most relevant document chunks..

# Context Injection & LLM Generation (Augmentation Step)
**What happens:**
* **Stuffing the Context:** The retrieved document chunks are concatenated (qdocs) and passed alongside the original question into the LLM (ChatOpenAI or OpenAI).

* **Generating the Answer:** The LLM reads the injected context and generates a structured answer (such as a Markdown table summarizing the matching shirts).

**Goal** : Question-Answering (Q&A) over private documents using Retrieval-Augmented Generation (RAG)

In [3]:
import os
import dotenv

In [4]:
llm_model = "gpt-3.5-turbo"

In [ ]:
from langchain.chains import RetrievelQA # It automates fetching relevant context from a vector store retriever and passing both context and query into an LLM to formulate an answer.
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import CSVLoader #his is used to load the csv files, formats it into individual Document objects that LangChain can process.
from langchain.vectorstores import DocArrayInMemorySearch # in-memory vector database. It converts document chunks into mathematical representation vectors (embeddings) and acts as a searchable index to find documents semantically similar to a user's query.

In [ ]:
file = 'yourfile.csv'
loader = CSVLoader(file_path=file)

In [ ]:
from langchain.indexes import VectorstoreIndexCreator

In [ ]:
index = VectorstoreIndexCreator(vectorstore_cls=DocArrayInMemorySearch).from_loaders([loader]) #Every vectors in the vector database are assigned with the index to retrieval the relavant information by calculatng the vector distance

In [ ]:
query ="Please list all your shirts with sun protection \in a table in markdown and summarize each one."

In [ ]:
llm_replacement_model = OpenAI(temperature=0, model='gpt-3.5-turbo-instruct')
response = index.query(query, llm = llm_replacement_model)

In [ ]:
display(Markdown(response))

**Output** : 

Name	Description	Sun Protection Rating
Men's Tropical Plaid Short-Sleeve Shirt	Made of 100% polyester, UPF 50+ rating, wrinkle-resistant, front and back cape venting, two front bellows pockets	SPF 50+, blocks 98% of harmful UV rays
Men's Plaid Tropic Shirt, Short-Sleeve	Made of 52% polyester and 48% nylon, UPF 50+ rating, SunSmart technology, wrinkle-free, front and back cape venting, two front bellows pockets	SPF 50+, blocks 98% of harmful UV rays
Men's TropicVibe Shirt, Short-Sleeve	Made of 71% nylon and 29% polyester, UPF 50+ rating, front and back cape venting, two front bellows pockets	SPF 50+, blocks 98% of harmful UV rays
Sun Shield Shirt	Made of 78% nylon and 22% Lycra Xtra Life fiber, UPF 50+ rating, moisture-wicking, abrasion-resistant, fits over swimsuit	SPF 50+, blocks 98% of harmful UV rays

## Step By Step

In [ ]:
from langchain.document_loader import CSVLoader
loader = CSVLoader(file_path=file

In [ ]:
docs = loader.load()

In [ ]:
from langchain.embeddings import OpenAIEmbedding
embedding = OpenAIEmbedding()

In [ ]:
embed = embeddings.embed_query("Hi my name is Harrison")

In [ ]:
print(len(embed))

**Output** : 1536

In [ ]:
db = DocArrayInMemorySearch(docs, embedding) 

Here db is the vector database where we pass the doc data and embedding, where it will convert the text data into chunks and into the mathematical vectors and store them into the vector database with index

In [7]:
query = "Please suggest a shirt with sunblocking"

In [ ]:
docs = db.similarity_search(query)

In [ ]:
retriever = db.as_retriever()

In [ ]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [ ]:
qdocs = "".join([docs[i].page_content for i in range(len(docs))]

In [ ]:
response = llm.call_as_llm(f"{qdocs} Question: Please list all your \shirts with sun protection in a table in markdown and summarize each one.") 

In [ ]:
qa_stuff = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=retriever, 
    verbose=True
)

In [ ]:
query =  "Please list all your shirts with sun protection in a table \in markdown and summarize each one."

In [ ]:
response = qa_stuff.run(query) 

**RetrievalQA** chain takes those retrieved document chunks ("stuffing" them as context) alongside the query prompt and feeds them into ChatOpenAI. The LLM uses this injected context to return a accurate answer (formatted in Markdown)

In [ ]:
display(Markdown(response)) #The output will be printed from this cell